# Recorrido del Código del Proyecto SimpleMDB

Este cuaderno proporciona un recorrido completo del proyecto SimpleMDB, siguiendo las solicitudes del cliente a través de todos los componentes, middleware y capas desde el cliente hasta la respuesta del servidor.

## Visión General del Proyecto

SimpleMDB es una API de base de datos de películas construida con C# usando HttpListener. Sigue una arquitectura por capas:

- **Capa de Presentación:** HttpServer, HttpRouter, Middleware
- **Capa de Aplicación:** Controladores (Controllers), Servicios (Services)
- **Capa de Dominio:** Modelos (Models), Lógica de Negocio
- **Capa de Infraestructura:** Repositorios, Base de Datos

## Diagrama de Arquitectura

```
Solicitud del Cliente
    ↓
HttpServer (App.cs)
    ↓
HttpRouter (Shared.Http)
    ↓
Middleware (p.ej., CORS, Manejo de Errores, Logging)
    ↓
MoviesRouter (Rutas a Controladores)
    ↓
MoviesController (Maneja verbos HTTP)
    ↓
IMovieService (Lógica de Negocio)
    ↓
IMovieRepository (Acceso a Datos)
    ↓
MemoryDatabase (Almacenamiento en Memoria)
    ↓
Respuesta del Servidor
```

## 1. Inicio de la solicitud del cliente

Un cliente (por ejemplo, Postman, cURL, navegador) envía una solicitud HTTP al servidor.

**Solicitud de ejemplo:** `GET http://localhost:3000/api/v1/movies/1`

- **Método:** GET
- **URL:** /api/v1/movies/1
- **Cabeceras:** Accept: application/json

## 2. HttpServer (App.cs)

El punto de entrada es `App.cs`, que hereda de `HttpServer` en `Shared.Http`.

```csharp
public class App : HttpServer
{
    public override void Init()
    {
        // Initialize dependencies: DB, Repo, Service, Controller, Router
        var db = new MemoryDatabase();
        var movieRepo = new MemoryMovieRepository(db);
        var movieServ = new DefaultMovieService(movieRepo);
        var movieCtrl = new MoviesController(movieServ);
        var movieRouter = new MoviesRouter(movieCtrl);
        
        // Configure middleware
        router.Use(HttpUtils.StructuredLogging);
        router.Use(HttpUtils.CentralizedErrorHandling);
        router.Use(HttpUtils.AddResponseCorsHeaders);
        router.Use(HttpUtils.DefaultResponse);
        router.Use(HttpUtils.ParseRequestUrl);
        router.Use(HttpUtils.ParseRequestQueryString);
        router.UseParametrizedRouteMatching();
        
        // Route to API
        router.UseRouter("/api/v1", apiRouter);
        apiRouter.UseRouter("/movies", movieRouter);
    }
}
```

El constructor de `HttpServer` configura el `HttpListener` en el host/puerto configurado (por ejemplo, http://localhost:3000).

En `Start()`, escucha solicitudes de forma asíncrona.

## 3. HttpRouter y Middleware

La solicitud se pasa a `HttpRouter.HandleAsync()`.

El middleware se aplica en orden:

1. **StructuredLogging:** Registra la solicitud.
2. **CentralizedErrorHandling:** Envuelve la ejecución en try-catch.
3. **AddResponseCorsHeaders:** Agrega encabezados CORS.
4. **DefaultResponse:** Establece una respuesta predeterminada si no hay ninguna.
5. **ParseRequestUrl:** Analiza la URL en componentes.
6. **ParseRequestQueryString:** Analiza los parámetros de consulta.
7. **UseParametrizedRouteMatching:** Habilita rutas parametrizadas como /:id.

Luego, se emparejan las rutas:
- `/api/v1` → apiRouter
- `/movies` → movieRouter
- `/1` → Coincide con `/:id` en MoviesRouter para GET.

## 4. MoviesRouter

```csharp
public class MoviesRouter : HttpRouter
{
    public MoviesRouter(MoviesController moviesController)
    {
        UseParametrizedRouteMatching();
        MapGet("/", moviesController.ReadMovies);
        MapPost("/", HttpUtils.ReadRequestBodyAsText, moviesController.CreateMovie);
        MapGet("/:id", moviesController.ReadMovie);
        MapPut("/:id", HttpUtils.ReadRequestBodyAsText, moviesController.UpdateMovie);
        MapDelete("/:id", moviesController.DeleteMovie);
    }
}
```

Para `GET /api/v1/movies/1`:
- Coincide con `MapGet("/:id", moviesController.ReadMovie)`
- Extrae `id = 1` en `props["req.params"]`
- Llama a `moviesController.ReadMovie()`

## 5. MoviesController

Maneja la solicitud HTTP y la delega al servicio.

```csharp
public async Task ReadMovie(HttpListenerRequest req, HttpListenerResponse res, Hashtable props, Func<Task> next)
{
    var uParams = (NameValueCollection)props["req.params"]!;
    int id = int.TryParse(uParams["id"]!, out int i) ? i : -1;
    var result = await movieService.ReadMovie(id);
    await JsonUtils.SendResultResponse(req, res, props, result);
    await next();
}
```

- Parsea el `id` desde los parámetros de URL.
- Llama a `movieService.ReadMovie(id)`.
- Envía la respuesta mediante `JsonUtils.SendResultResponse()`. 

## 6. IMovieService (DefaultMovieService)

Capa de lógica de negocio.

```csharp
public async Task<Result<Movie>> ReadMovie(int id)
{
    var movie = await movieRepository.ReadMovie(id);
    var result = movie == null
        ? new Result<Movie>(new Exception($"Could not read movie with id {id}."), (int)HttpStatusCode.NotFound)
        : new Result<Movie>(movie, (int)HttpStatusCode.OK);
    return result;
}
```

- Llama al repositorio para obtener datos.
- Envuelve el resultado en `Result<T>` con un código de estado.

## 7. IMovieRepository (MemoryMovieRepository)

Capa de acceso a datos.

```csharp
public async Task<Movie?> ReadMovie(int id)
{
    Movie? result = db.Movies.FirstOrDefault(m => m.Id == id);
    return await Task.FromResult(result);
}
```

- Consulta la base de datos en memoria.
- Devuelve la película o null.

## 8. MemoryDatabase

Almacenamiento en memoria inicializado con 50 películas.

```csharp
public class MemoryDatabase
{
    public List<Movie> Movies { get; }
    // Seeded in constructor
}
```

- Almacena datos en una `List<Movie>`.
- No persiste; los datos se reinician al reiniciar la aplicación.

## 9. Flujo de respuesta de regreso

- El repositorio devuelve `Movie` o null.
- El servicio envuelve en `Result<Movie>` (OK o NotFound).
- El controlador llama a `JsonUtils.SendResultResponse()`, que:
  - Establece el código de estado.
  - Serializa el resultado a JSON.
  - Escribe en el flujo de respuesta.
- El middleware maneja cualquier procesamiento final (por ejemplo, logging).
- HttpServer envía la respuesta al cliente.

**Ejemplo de respuesta para GET /api/v1/movies/1:**
```json
{
  "id": 1,
  "title": "The Godfather",
  "year": 1972,
  "description": "A mafia patriarch hands the family empire to his reluctant son."
}
```

## Ejemplo: POST /api/v1/movies (Crear)

1. **Cliente:** Envía un cuerpo JSON con los datos de la película.
2. **Middleware:** `ReadRequestBodyAsText` lee el cuerpo y lo guarda en `props["req.text"]`.
3. **Controlador:** Deserializa JSON a `Movie`, llama a `service.CreateMovie(movie)`.
4. **Servicio:** Valida los datos, llama a `repo.CreateMovie(movie)` (asigna un nuevo ID).
5. **Repositorio:** Agrega a `db.Movies`, devuelve la película.
6. **Respuesta:** 201 Created con el JSON de la película nueva.

## Ejemplo: PUT /api/v1/movies/1 (Actualizar)

Similar a Crear, pero:
- Se lee y deserializa el cuerpo.
- El servicio valida y llama a `repo.UpdateMovie(id, newData)`.
- El repositorio encuentra el existente y actualiza campos.
- Respuesta: 200 OK con la película actualizada.

## Ejemplo: DELETE /api/v1/movies/1

- Sin cuerpo.
- El controlador llama a `service.DeleteMovie(id)`.
- El repositorio elimina de la lista y devuelve la película eliminada.
- Respuesta: 200 OK con la película eliminada.

## Manejo de Errores

- JSON inválido: la deserialización falla y el servicio devuelve 400.
- ID inexistente: el repositorio devuelve null y el servicio devuelve 404.
- Errores de validación (por ejemplo, título vacío): el servicio devuelve 400.

## Componentes Clave

- **Shared.Http:** Utilidades HTTP reutilizables (Router, Middleware, Utils).
- **Smdb.Core:** Modelos de dominio, servicios, repositorios.
- **Smdb.Api:** Controladores, enrutamiento, inicio de la aplicación.

Este recorrido cubre el ciclo completo de solicitud-respuesta. Estudia cada capa para comprender la separación de responsabilidades.